In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt

In [ ]:
BATCH_SIZE = 16
IMAGE_SIZE = 256
CHANNELS=3

In [ ]:
import os
from PIL import Image

# Define the path to your image directory
image_directory = "/content/drive/MyDrive/Pictures"

# Get a list of all image files in the directory
image_files = [f for f in os.listdir(image_directory) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

# Loop through image files
for image_file in image_files:
    image_path = os.path.join(image_directory, image_file)

    try:
        # Try to open the image using PIL
        image = Image.open(image_path)
        image.verify()  # Verify that the image is not corrupted
        print(f"Image '{image_file}' is valid.")
    except Exception as e:
        print(f"Image '{image_file}' is corrupted or unsupported: {str(e)}")

In [ ]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/drive/MyDrive/Pictures",
    seed=12,
    shuffle=True,
    image_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

In [ ]:
class_names = dataset.class_names
class_names

In [ ]:
len(dataset)

In [ ]:
int(len(dataset)*0.8)

In [ ]:
train_ds = dataset.take(12)
len(train_ds)

In [ ]:
test_ds = dataset.skip(12)
len(test_ds)

In [ ]:
val_size=0.1
(len(dataset)*val_size)

In [ ]:
val_ds = test_ds.take(2)
len(val_ds)

In [ ]:
test_ds = test_ds.skip(2)
len(test_ds)

In [ ]:
def get_dataset_partitions_tf(data, train_split=0.8, val_split=0.1, test_split=0.1, shuffle=True, shuffle_size=10000):
    assert (train_split + test_split + val_split) == 1

    data_size = len(data)

    if shuffle:
        data = data.shuffle(shuffle_size, seed=12)

    train_size = int(train_split * data_size)
    val_size = int(val_split * data_size)

    train_ds = data.take(train_size)
    val_ds = data.skip(train_size).take(val_size)
    test_ds = data.skip(train_size).skip(val_size)

    return train_ds, val_ds, test_ds

In [ ]:
train_ds, val_ds, test_ds = get_dataset_partitions_tf(dataset)

In [ ]:
len(train_ds)

In [ ]:
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
resize_and_rescale = tf.keras.Sequential([
  layers.experimental.preprocessing.Resizing(IMAGE_SIZE, IMAGE_SIZE),
  layers.experimental.preprocessing.Rescaling(1./255),
])

In [ ]:
data_augmentation = tf.keras.Sequential([
  layers.experimental.preprocessing.RandomFlip("horizontal_and_vertical"),
  layers.experimental.preprocessing.RandomRotation(0.2),
])

In [ ]:
import os
def augment_data(x, y):
    return (data_augmentation(x, training=True), y)

def data_augmentation(image, training):
    image = tf.cast(image, tf.uint8)
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.2, upper=1.8)
    return image


In [ ]:

if not os.path.exists('augmented_data'):
    os.makedirs('augmented_data')

image_dir = 'augmented_data/images'

if not os.path.exists(image_dir):
    os.makedirs(image_dir)

In [ ]:
!pip install Pillow


In [ ]:
from PIL import Image
import numpy as np

for image_path, label in train_ds:
    image = Image.open(image_path.numpy().decode('utf-8'))  # Decode image path to string
    image = np.array(image)  # Convert PIL Image to NumPy array
    print(f"Image shape: {image.shape}, Label: {label}")
    # Rest of your code
